<a href="https://colab.research.google.com/github/trungkien1502/project_sna_group14/blob/main/02_E1_LightGCN_Setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# ==========================================
# CELL 1: GPU & GOOGLE DRIVE SETUP
# ==========================================

import sys
import torch
from pathlib import Path
from google.colab import drive

# 1. Kiểm tra môi trường
print("=== ENVIRONMENT ===")
print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), "GPU chưa được bật!"

print("GPU:", torch.cuda.get_device_name(0))

# 2. Kết nối Google Drive
drive.mount("/content/drive")

# 3. Xác định thư mục dữ liệu
ROOT = Path("/content/drive/MyDrive/Data_SNA_Group14")
DATA_DIR = ROOT / "data/raw/amazon"

assert DATA_DIR.exists(), "Không tìm thấy dataset!"

print("\n=== SETUP SUCCESS ===")
print("Dataset:", DATA_DIR)

=== ENVIRONMENT ===
Python: 3.13.15
PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

=== SETUP SUCCESS ===
Dataset: /content/drive/MyDrive/Data_SNA_Group14/data/raw/amazon


In [6]:
# ==========================================
# CELL 2: RLMRec REPOSITORY & DATASET
# ==========================================

import subprocess
from pathlib import Path

REPO = Path("/content/RLMRec")

# 1. Clone repository tác giả
if not REPO.exists():
    subprocess.run(
        [
            "git", "clone", "--depth", "1",
            "https://github.com/HKUDS/RLMRec.git",
            str(REPO)
        ],
        check=True
    )

# 2. Ghi nhận phiên bản RLMRec
commit = subprocess.check_output(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    text=True
).strip()

print("RLMRec commit:", commit)

# 3. Kiểm tra dataset
required_files = [
    "trn_mat.pkl",
    "val_mat.pkl",
    "tst_mat.pkl",
    "usr_prf.pkl",
    "itm_prf.pkl",
    "usr_emb_np.pkl",
    "itm_emb_np.pkl"
]

for filename in required_files:
    assert (DATA_DIR / filename).is_file(), (
        f"Missing: {filename}"
    )

# 4. Kết nối dữ liệu từ Drive vào RLMRec
target = REPO / "data" / "amazon"

if not target.exists() and not target.is_symlink():
    target.symlink_to(
        DATA_DIR,
        target_is_directory=True
    )

assert target.resolve() == DATA_DIR.resolve(), (
    "Dataset link đang trỏ sai!"
)

print("\n=== DATASET CONNECTED ===")
print("Repository:", REPO)
print("Dataset:", target.resolve())
print("Training script:", (REPO / "encoder/train_encoder.py").exists())

RLMRec commit: 22413752246de3dee8ab0d509f7f7a8889080f95

=== DATASET CONNECTED ===
Repository: /content/RLMRec
Dataset: /content/drive/MyDrive/Data_SNA_Group14/data/raw/amazon
Training script: True


In [7]:
# ==========================================
# CELL 3: LIGHTGCN SMOKE TEST
# ==========================================

import sys
import subprocess
import json
from pathlib import Path

def run_smoke(model_name, output_folder):

    OUT = ROOT / "experiments/E1" / output_folder
    OUT.mkdir(parents=True, exist_ok=True)

    smoke_code = """
import sys
import json

sys.path.insert(0, "./encoder")

from config.configurator import configs
from trainer.trainer import init_seed
from data_utils.build_data_handler import build_data_handler
from models.bulid_model import build_model
from trainer.logger import Logger
from trainer.build_trainer import build_trainer

# Initialize seed
init_seed()

# Load dataset
handler = build_data_handler()
handler.load_data()

# Build model
model = build_model(handler).to(configs["device"])

# Initialize trainer
logger = Logger()
trainer = build_trainer(handler, logger)
trainer.create_optimizer(model)

# Train exactly 1 epoch
print("SMOKE_TEST: TRAIN_START", flush=True)

trainer.train_epoch(model, 0)

print("SMOKE_TEST: TRAIN_SUCCESS", flush=True)

# Validation only
print("SMOKE_TEST: VALIDATION_START", flush=True)

metrics = trainer.evaluate(model, 0)

result = {
    "model": configs["model"]["name"],
    "dataset": "amazon",
    "seed": configs["train"]["seed"],
    "epochs_completed": 1,
    "evaluation_split": "validation",
    "smoke_test": True,
    "metrics": {
        k: [float(v) for v in values]
        for k, values in metrics.items()
    },
    "k": configs["test"]["k"]
}

print(
    "SMOKE_RESULT_JSON=" + json.dumps(result),
    flush=True
)

print("SMOKE_TEST: SUCCESS", flush=True)
"""

    cmd = [
        sys.executable, "-c", smoke_code,
        "--model", model_name,
        "--dataset", "amazon",
        "--cuda", "0",
        "--seed", "2023"
    ]

    print(f"Running: {model_name}")

    proc = subprocess.run(
        cmd,
        cwd=str(REPO),
        text=True,
        capture_output=True
    )

    # Save logs
    (OUT / "smoke_test.log").write_text(
        proc.stdout + "\n--- STDERR ---\n" + proc.stderr,
        encoding="utf-8"
    )

    print("Return code:", proc.returncode)
    print(proc.stdout[-4000:])

    if proc.returncode != 0:
        print("\n=== STDERR ===")
        print(proc.stderr[-5000:])
        raise RuntimeError(f"{model_name} smoke test failed")

    # Extract metrics
    prefix = "SMOKE_RESULT_JSON="

    line = next(
        (x for x in proc.stdout.splitlines()
         if x.startswith(prefix)),
        None
    )

    assert line is not None, "Missing result JSON"

    result = json.loads(line[len(prefix):])

    # Save metrics
    metrics_file = OUT / "smoke_metrics.json"

    metrics_file.write_text(
        json.dumps(result, indent=2),
        encoding="utf-8"
    )

    print(f"\n=== {model_name} SMOKE TEST COMPLETE ===")
    print("Metrics saved:", metrics_file)

    return result


# Run LightGCN
lightgcn_result = run_smoke(
    model_name="lightgcn",
    output_folder="smoke_lightgcn"
)

Running: lightgcn
Return code: 0
{'optimizer': {'name': 'adam', 'lr': 0.001, 'weight_decay': 0}, 'train': {'epoch': 3000, 'batch_size': 4096, 'save_model': False, 'loss': 'pairwise', 'test_step': 3, 'reproducible': True, 'seed': 2023, 'patience': 5}, 'test': {'metrics': ['recall', 'ndcg'], 'k': [5, 10, 20], 'batch_size': 1024}, 'data': {'type': 'general_cf', 'name': 'amazon', 'user_num': 11000, 'item_num': 9332}, 'model': {'name': 'lightgcn', 'keep_rate': 0.8, 'embedding_size': 32, 'layer_num': 3, 'reg_weight': 1e-06, 'amazon': {'layer_num': 3, 'reg_weight': 1e-06}, 'yelp': {'layer_num': 3, 'reg_weight': 1e-06}, 'steam': {'layer_num': 2, 'reg_weight': 1e-06}}}
SMOKE_TEST: TRAIN_START
SMOKE_TEST: TRAIN_SUCCESS
SMOKE_TEST: VALIDATION_START
Epoch   0 Validation set [recall@5: 0.0191 recall@10: 0.0323 recall@20: 0.0527 ] [ndcg@5: 0.0187 ndcg@10: 0.0235 ndcg@20: 0.0302 ] 
SMOKE_RESULT_JSON={"model": "lightgcn", "dataset": "amazon", "seed": 2023, "epochs_completed": 1, "evaluation_split": "v

In [8]:
# ==========================================
# CELL 4: RLMRec-Con SMOKE TEST
# ==========================================

rlmrec_result = run_smoke(
    model_name="lightgcn_plus",
    output_folder="smoke_rlmrec_con"
)

print("\n=== BOTH MODELS VERIFIED ===")

for name, result in [
    ("LightGCN", lightgcn_result),
    ("RLMRec-Con", rlmrec_result)
]:
    ks = result["k"]
    recall = result["metrics"]["recall"]
    ndcg = result["metrics"]["ndcg"]

    idx = ks.index(20)

    print(f"\n{name}")
    print(f"Recall@20: {recall[idx]:.4f}")
    print(f"NDCG@20:   {ndcg[idx]:.4f}")

print("\nE1 ENVIRONMENT SMOKE TEST: PASSED")

Running: lightgcn_plus
Return code: 0
{'optimizer': {'name': 'adam', 'lr': 0.001, 'weight_decay': 0}, 'train': {'epoch': 3000, 'batch_size': 4096, 'save_model': True, 'loss': 'pairwise', 'test_step': 3, 'reproducible': True, 'seed': 2023, 'patience': 5}, 'test': {'metrics': ['recall', 'ndcg'], 'k': [5, 10, 20], 'batch_size': 1024}, 'data': {'type': 'general_cf', 'name': 'amazon', 'user_num': 11000, 'item_num': 9332}, 'model': {'name': 'lightgcn_plus', 'keep_rate': 0.8, 'embedding_size': 32, 'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, 'kd_temperature': 0.2, 'amazon': {'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, 'kd_temperature': 0.2}, 'yelp': {'layer_num': 3, 'reg_weight': 1e-07, 'kd_weight': 0.01, 'kd_temperature': 0.2}, 'steam': {'layer_num': 3, 'reg_weight': 1e-08, 'kd_weight': 0.001, 'kd_temperature': 0.1}}}
SMOKE_TEST: TRAIN_START
SMOKE_TEST: TRAIN_SUCCESS
SMOKE_TEST: VALIDATION_START
Epoch   0 Validation set [recall@5: 0.0311 recall@10: 0.0511 recall@20: 0.